# 静岡県3都市の道路ネットワーク比較分析

焼津市・静岡市・浜松市の道路ネットワークをOSMnxで取得し、8指標を用いて定量的に比較する。
劉（2016）の道路ネットワーク解析手法に基づき、回遊性・アクセス性・迂回性の3観点から構造化して分析する。

- **防災観点**: 沿岸部vs内陸部の避難路ネットワーク特性
- **観光観点**: 駅周辺の歩行者ネットワーク回遊性

実験計画: `docs/plans/exp001_shizuoka_road_network_comparison.md`

## Section 1: 環境セットアップ

### 1.1 パッケージインストール

In [ ]:
%%capture
!pip install osmnx networkx geopandas folium matplotlib japanize-matplotlib contextily shapely scipy tqdm

### 1.2 Google Driveマウント & パス設定

In [ ]:
import os
import sys

# Colab環境の場合はGoogle Driveをマウント
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/geo-laboratory'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)
print(f"プロジェクトルート: {PROJECT_ROOT}")

### 1.3 ライブラリのインポート

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
import matplotlib.pyplot as plt
import japanize_matplotlib
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import nearest_points, unary_union
from tqdm.auto import tqdm

from src.network_metrics import NetworkMetricsCalculator
from src.visualization import (
    plot_network_on_map,
    plot_radar_chart,
    plot_comparison_bars,
    plot_subarea_comparison,
    plot_liu_typology_comparison,
)

# OSMnxの設定
ox.settings.use_cache = True
ox.settings.log_console = True

# 指標計算クラスのインスタンス化
calc = NetworkMetricsCalculator()

print(f"OSMnx version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")

## Section 2: データ取得

### 2.1 3都市の道路ネットワーク取得（drive）

In [ ]:
# 対象3都市の定義
cities = {
    "焼津市": "焼津市, 静岡県, Japan",
    "静岡市": "静岡市, 静岡県, Japan",
    "浜松市": "浜松市, 静岡県, Japan",
}

# 道路ネットワークの取得と投影変換
city_graphs = {}
city_graphs_projected = {}

for name, query in tqdm(cities.items(), desc="都市ネットワーク取得"):
    print(f"\n--- {name} ---")
    G = ox.graph_from_place(query, network_type='drive')
    city_graphs[name] = G
    G_proj = ox.project_graph(G)
    city_graphs_projected[name] = G_proj
    print(f"  ノード数: {G_proj.number_of_nodes():,}")
    print(f"  エッジ数: {G_proj.number_of_edges():,}")

### 2.2 サブエリア抽出（沿岸部2km / 内陸部）

In [ ]:
# 各都市の行政区域ポリゴンを取得
city_boundaries = {}
for name, query in tqdm(cities.items(), desc="行政区域取得"):
    gdf = ox.geocode_to_gdf(query)
    city_boundaries[name] = gdf
    print(f"{name}: 面積 {gdf.to_crs(epsg=6677).area.values[0] / 1e6:.1f} km²")

In [ ]:
def get_coastline_for_city(city_name, boundary_gdf):
    """都市の海岸線ジオメトリを取得する"""
    try:
        # OSMから海岸線を取得
        coastline = ox.features_from_polygon(
            boundary_gdf.geometry.values[0],
            tags={'natural': 'coastline'}
        )
        if len(coastline) > 0:
            print(f"{city_name}: 海岸線 {len(coastline)} フィーチャー取得")
            return coastline
    except Exception as e:
        print(f"{city_name}: 海岸線取得エラー - {e}")

    # 海岸線が取得できない場合はwaterの境界を代替利用
    try:
        water = ox.features_from_polygon(
            boundary_gdf.geometry.values[0],
            tags={'natural': 'water'}
        )
        if len(water) > 0:
            print(f"{city_name}: 水域 {len(water)} フィーチャーを代替取得")
            return water
    except Exception as e:
        print(f"{city_name}: 水域取得エラー - {e}")

    return None


# 海岸線データの取得
coastlines = {}
for name in cities:
    coastlines[name] = get_coastline_for_city(name, city_boundaries[name])

In [ ]:
COASTAL_THRESHOLD_M = 2000  # 沿岸部の閾値（メートル）

def split_graph_by_coastline(G_proj, coastline_gdf, threshold_m=COASTAL_THRESHOLD_M):
    """グラフを海岸線からの距離で沿岸部と内陸部に分割する

    Parameters
    ----------
    G_proj : networkx.MultiDiGraph
        投影座標系の道路ネットワーク
    coastline_gdf : GeoDataFrame
        海岸線のジオメトリ
    threshold_m : float
        沿岸部の閾値距離（メートル）

    Returns
    -------
    tuple
        (沿岸部サブグラフ, 内陸部サブグラフ, ノードごとの距離Series)
    """
    nodes_gdf = ox.graph_to_gdfs(G_proj, edges=False)
    crs = G_proj.graph['crs']

    # 海岸線を投影座標系に変換し統合
    coastline_proj = coastline_gdf.to_crs(crs)
    coastline_union = unary_union(coastline_proj.geometry)

    # 各ノードから海岸線までの距離を計算
    distances = nodes_gdf.geometry.distance(coastline_union)

    # 沿岸部・内陸部のノードを分類
    coastal_nodes = distances[distances <= threshold_m].index.tolist()
    inland_nodes = distances[distances > threshold_m].index.tolist()

    # サブグラフの抽出
    G_coastal = G_proj.subgraph(coastal_nodes).copy() if coastal_nodes else None
    G_inland = G_proj.subgraph(inland_nodes).copy() if inland_nodes else None

    return G_coastal, G_inland, distances


# 各都市について沿岸部/内陸部を分割
coastal_graphs = {}
inland_graphs = {}
node_distances = {}

for name in cities:
    if coastlines[name] is not None and len(coastlines[name]) > 0:
        G_coastal, G_inland, dists = split_graph_by_coastline(
            city_graphs_projected[name], coastlines[name]
        )
        coastal_graphs[name] = G_coastal
        inland_graphs[name] = G_inland
        node_distances[name] = dists

        n_coastal = G_coastal.number_of_nodes() if G_coastal else 0
        n_inland = G_inland.number_of_nodes() if G_inland else 0
        print(f"{name}: 沿岸部 {n_coastal:,} ノード / 内陸部 {n_inland:,} ノード")
    else:
        print(f"{name}: 海岸線データなし（スキップ）")
        coastal_graphs[name] = None
        inland_graphs[name] = None

### 2.3 駅周辺800mネットワーク取得（walk + drive）

In [ ]:
# 対象駅の定義
stations = {
    "焼津駅": (34.8671, 138.3225),
    "静岡駅": (34.9717, 138.3891),
    "浜松駅": (34.7038, 137.7350),
}

STATION_RADIUS = 800  # メートル

# 歩行者ネットワーク（walk: 細街路を含む）
station_graphs_walk = {}
station_graphs_walk_projected = {}

# 自動車ネットワーク（drive: 主要道路のみ）
station_graphs_drive = {}
station_graphs_drive_projected = {}

for name, (lat, lon) in tqdm(stations.items(), desc="駅周辺ネットワーク取得"):
    print(f"\n--- {name} ---")

    # Walk
    G_walk = ox.graph_from_point((lat, lon), dist=STATION_RADIUS, network_type='walk')
    station_graphs_walk[name] = G_walk
    G_walk_proj = ox.project_graph(G_walk)
    station_graphs_walk_projected[name] = G_walk_proj
    print(f"  Walk - ノード数: {G_walk_proj.number_of_nodes():,}, エッジ数: {G_walk_proj.number_of_edges():,}")

    # Drive
    G_drive = ox.graph_from_point((lat, lon), dist=STATION_RADIUS, network_type='drive')
    station_graphs_drive[name] = G_drive
    G_drive_proj = ox.project_graph(G_drive)
    station_graphs_drive_projected[name] = G_drive_proj
    print(f"  Drive - ノード数: {G_drive_proj.number_of_nodes():,}, エッジ数: {G_drive_proj.number_of_edges():,}")

### 2.4 基本統計量の確認

In [ ]:
# 市全域の基本統計
basic_stats = []
for name, G in city_graphs_projected.items():
    basic_stats.append({
        '都市': name,
        'ノード数': G.number_of_nodes(),
        'エッジ数': G.number_of_edges(),
        '連結成分数': nx.number_weakly_connected_components(G),
    })

df_basic = pd.DataFrame(basic_stats)
display(df_basic)

## Section 3: 市全域の指標算出と劉の3観点による構造化

### 3.1 各市の8指標を算出

In [ ]:
city_metrics = {}

for name, G in tqdm(city_graphs_projected.items(), desc="市全域指標算出"):
    print(f"\n--- {name} ---")
    metrics = calc.calculate_all(G)
    city_metrics[name] = metrics
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

### 3.2 結果をDataFrameにまとめて表示

In [ ]:
df_city_metrics = pd.DataFrame(city_metrics).T
df_city_metrics.index.name = '都市'

# 劉の3観点でグルーピングして表示
print("=== 回遊性指標 ===")
display(df_city_metrics[calc.CIRCULATION_INDICATORS])

print("\n=== アクセス性指標 ===")
display(df_city_metrics[calc.ACCESSIBILITY_INDICATORS])

print("\n=== 迂回性指標 ===")
display(df_city_metrics[calc.CIRCUITY_INDICATORS])

### 3.3 劉の3観点（回遊性・アクセス性・迂回性）別のグループ棒グラフ

In [ ]:
fig = plot_comparison_bars(city_metrics, title="市全域: 劉の3観点別指標比較")
plt.show()

### 3.4 レーダーチャートで3都市比較

In [ ]:
fig = plot_radar_chart(city_metrics, title="市全域: 3都市8指標レーダーチャート")
plt.show()

## Section 4: 防災分析（沿岸部 vs 内陸部）

### 4.1 沿岸部・内陸部の8指標を算出

In [ ]:
coastal_metrics = {}
inland_metrics = {}

for name in cities:
    print(f"\n=== {name} ===")

    # 沿岸部
    if coastal_graphs.get(name) is not None and coastal_graphs[name].number_of_nodes() > 5:
        print(f"  沿岸部（{coastal_graphs[name].number_of_nodes():,} ノード）...")
        coastal_metrics[name] = calc.calculate_all(coastal_graphs[name])
    else:
        print("  沿岸部: ノード不足でスキップ")
        coastal_metrics[name] = None

    # 内陸部
    if inland_graphs.get(name) is not None and inland_graphs[name].number_of_nodes() > 5:
        print(f"  内陸部（{inland_graphs[name].number_of_nodes():,} ノード）...")
        inland_metrics[name] = calc.calculate_all(inland_graphs[name])
    else:
        print("  内陸部: ノード不足でスキップ")
        inland_metrics[name] = None

### 4.2 回遊性指標の比較

劉の知見: 沿岸部は類型IIに近く回遊性が低い傾向があるか

In [ ]:
# 沿岸部 vs 内陸部の回遊性指標を比較
print("=== 回遊性指標: 沿岸部 vs 内陸部 ===")
for name in cities:
    print(f"\n{name}:")
    for ind in calc.CIRCULATION_INDICATORS:
        c_val = coastal_metrics[name][ind] if coastal_metrics[name] else float('nan')
        i_val = inland_metrics[name][ind] if inland_metrics[name] else float('nan')
        diff = c_val - i_val if not (np.isnan(c_val) or np.isnan(i_val)) else float('nan')
        print(f"  {ind}: 沿岸部={c_val:.4f}, 内陸部={i_val:.4f}, 差={diff:+.4f}")

### 4.3 迂回性指標の比較

劉の知見: 海岸線付近の急峻な地形が迂回率を増大させるか

In [ ]:
# 沿岸部 vs 内陸部の迂回性指標を比較
print("=== 迂回性指標: 沿岸部 vs 内陸部 ===")
for name in cities:
    print(f"\n{name}:")
    for ind in calc.CIRCUITY_INDICATORS:
        c_val = coastal_metrics[name][ind] if coastal_metrics[name] else float('nan')
        i_val = inland_metrics[name][ind] if inland_metrics[name] else float('nan')
        diff = c_val - i_val if not (np.isnan(c_val) or np.isnan(i_val)) else float('nan')
        print(f"  {ind}: 沿岸部={c_val:.4f}, 内陸部={i_val:.4f}, 差={diff:+.4f}")

### 4.4 沿岸部ネットワークの地図可視化

In [ ]:
# 各都市の沿岸部ネットワークを地図に描画
for name in cities:
    if coastal_graphs.get(name) is not None and coastal_graphs[name].number_of_nodes() > 0:
        # 非投影座標系のグラフで地図描画
        G_orig = city_graphs[name]
        nodes_gdf = ox.graph_to_gdfs(city_graphs_projected[name], edges=False)

        # 沿岸部ノードのosmidを取得
        coastal_node_ids = list(coastal_graphs[name].nodes())

        # 元のグラフから沿岸部サブグラフを作成
        valid_ids = [n for n in coastal_node_ids if n in G_orig.nodes()]
        if valid_ids:
            G_coastal_orig = G_orig.subgraph(valid_ids).copy()
            m = plot_network_on_map(
                G_coastal_orig,
                title=f"{name} 沿岸部（海岸線から{COASTAL_THRESHOLD_M/1000:.0f}km以内）",
                metrics=coastal_metrics[name],
                color="red",
            )
            display(m)

### 4.5 棒グラフによる沿岸部 vs 内陸部の比較

In [ ]:
# 沿岸部 vs 内陸部の全指標比較
subarea_data = {}
for name in cities:
    subarea_data[name] = {}
    if coastal_metrics[name]:
        subarea_data[name]["沿岸部"] = coastal_metrics[name]
    if inland_metrics[name]:
        subarea_data[name]["内陸部"] = inland_metrics[name]

# データがある都市のみプロット
plot_data = {k: v for k, v in subarea_data.items() if len(v) == 2}
if plot_data:
    fig = plot_subarea_comparison(plot_data, area_type="coastal", title="防災分析: 沿岸部 vs 内陸部")
    plt.show()
else:
    print("沿岸部・内陸部の両方のデータが揃った都市がありません")

## Section 5: 観光分析（駅周辺の回遊性）

### 5.1 駅周辺800mの8指標を算出（walk + drive）

In [ ]:
station_metrics_walk = {}
station_metrics_drive = {}

for name in tqdm(stations.keys(), desc="駅周辺指標算出"):
    print(f"\n=== {name} ===")

    # Walk
    print("  Walk...")
    station_metrics_walk[name] = calc.calculate_all(
        station_graphs_walk_projected[name]
    )

    # Drive
    print("  Drive...")
    station_metrics_drive[name] = calc.calculate_all(
        station_graphs_drive_projected[name]
    )

### 5.2 細街路効果の分析: walk vs drive

劉の知見: 都市部（類型III）では細街路が袋路状で回遊性に寄与しにくい vs 漁港町では細街路が回遊性を向上させる

In [ ]:
# 細街路効果: walk - drive の差分
print("=== 細街路効果（walk - drive）===")
fine_street_effect = {}

for name in stations:
    print(f"\n{name}:")
    effect = {}
    for ind in calc.FEATURE_COLUMNS:
        w_val = station_metrics_walk[name].get(ind, 0)
        d_val = station_metrics_drive[name].get(ind, 0)
        diff = w_val - d_val
        effect[ind] = diff
        print(f"  {ind}: walk={w_val:.4f}, drive={d_val:.4f}, 差={diff:+.4f}")
    fine_street_effect[name] = effect

# 細街路効果のDataFrame
df_fine_effect = pd.DataFrame(fine_street_effect).T
df_fine_effect.index.name = '駅'
print("\n=== 細街路効果まとめ ===")
display(df_fine_effect)

In [ ]:
# 細街路効果の回遊性指標への影響を可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

station_names = list(stations.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# 回遊性指標の比較
ax = axes[0]
x = np.arange(len(calc.CIRCULATION_INDICATORS))
width = 0.25
for i, name in enumerate(station_names):
    vals = [fine_street_effect[name][ind] for ind in calc.CIRCULATION_INDICATORS]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])
ax.set_title('回遊性指標への細街路効果（walk - drive）', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(calc.CIRCULATION_INDICATORS, rotation=45, ha='right', fontsize=7)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.legend(fontsize=8)

# 迂回性指標の比較
ax = axes[1]
x = np.arange(len(calc.CIRCUITY_INDICATORS))
for i, name in enumerate(station_names):
    vals = [fine_street_effect[name][ind] for ind in calc.CIRCUITY_INDICATORS]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])
ax.set_title('迂回性指標への細街路効果（walk - drive）', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(calc.CIRCUITY_INDICATORS, rotation=45, ha='right', fontsize=7)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 5.3 アクセス性指標の比較

In [ ]:
# 駅周辺のアクセス性指標（walk）を比較
print("=== アクセス性指標: 駅周辺（walk） ===")
for ind in calc.ACCESSIBILITY_INDICATORS:
    print(f"\n{ind}:")
    for name in stations:
        val = station_metrics_walk[name].get(ind, 0)
        print(f"  {name}: {val:.4f}")

### 5.4 駅周辺ネットワークの地図可視化

In [ ]:
# 各駅周辺のネットワーク（walk）を地図に描画
for name in stations:
    m = plot_network_on_map(
        station_graphs_walk[name],
        title=f"{name}周辺{STATION_RADIUS}m 歩行者ネットワーク",
        metrics=station_metrics_walk[name],
        color="blue",
        weight=2,
    )
    display(m)

### 5.5 劉の4類型との対照

In [ ]:
# 劉（2016）の4類型の概念的プロファイル
# 劉の論文の知見に基づく相対的な特性値（0-1の正規化スケール）
# 実際の値ではなく、各類型の相対的な傾向を表現
liu_type_profiles = {
    "類型I（島嶼・山間部）": {
        'alpha_index': 0.2,
        'degree_centrality_mean': 0.3,
        'basic_streets_per_node_avg': 0.3,
        'basic_self_loop_proportion': 0.1,
        'closeness_centrality_mean': 0.2,
        'basic_clean_intersection_density_km': 0.2,
        'avg_circuity_A': 0.8,
        'basic_circuity_avg': 0.8,
    },
    "類型II（沿岸・近都市部）": {
        'alpha_index': 0.3,
        'degree_centrality_mean': 0.4,
        'basic_streets_per_node_avg': 0.4,
        'basic_self_loop_proportion': 0.15,
        'closeness_centrality_mean': 0.4,
        'basic_clean_intersection_density_km': 0.3,
        'avg_circuity_A': 0.6,
        'basic_circuity_avg': 0.6,
    },
    "類型III（都市部）": {
        'alpha_index': 0.7,
        'degree_centrality_mean': 0.7,
        'basic_streets_per_node_avg': 0.7,
        'basic_self_loop_proportion': 0.3,
        'closeness_centrality_mean': 0.8,
        'basic_clean_intersection_density_km': 0.8,
        'avg_circuity_A': 0.3,
        'basic_circuity_avg': 0.3,
    },
    "類型IV（寺内町）": {
        'alpha_index': 0.8,
        'degree_centrality_mean': 0.6,
        'basic_streets_per_node_avg': 0.6,
        'basic_self_loop_proportion': 0.05,
        'closeness_centrality_mean': 0.7,
        'basic_clean_intersection_density_km': 0.9,
        'avg_circuity_A': 0.2,
        'basic_circuity_avg': 0.2,
    },
}

# 駅周辺（walk）の指標と劉の4類型を対照
fig = plot_liu_typology_comparison(
    station_metrics_walk,
    liu_type_profiles,
    title="駅周辺歩行者ネットワーク vs 劉（2016）4類型",
)
plt.show()

## Section 6: 総合比較

### 6.1 全分析結果の統合テーブル

In [ ]:
# 全結果を統合
all_results = []

# 市全域
for name, metrics in city_metrics.items():
    row = {'都市': name, 'エリア': '市全域', 'タイプ': 'drive'}
    row.update(metrics)
    all_results.append(row)

# 沿岸部
for name, metrics in coastal_metrics.items():
    if metrics:
        row = {'都市': name, 'エリア': '沿岸部', 'タイプ': 'drive'}
        row.update(metrics)
        all_results.append(row)

# 内陸部
for name, metrics in inland_metrics.items():
    if metrics:
        row = {'都市': name, 'エリア': '内陸部', 'タイプ': 'drive'}
        row.update(metrics)
        all_results.append(row)

# 駅周辺（walk）
city_for_station = {'焼津駅': '焼津市', '静岡駅': '静岡市', '浜松駅': '浜松市'}
for name, metrics in station_metrics_walk.items():
    row = {'都市': city_for_station[name], 'エリア': f'{name}周辺', 'タイプ': 'walk'}
    row.update(metrics)
    all_results.append(row)

# 駅周辺（drive）
for name, metrics in station_metrics_drive.items():
    row = {'都市': city_for_station[name], 'エリア': f'{name}周辺', 'タイプ': 'drive'}
    row.update(metrics)
    all_results.append(row)

df_all = pd.DataFrame(all_results)
display(df_all)

### 6.2 ヒートマップによる指標×都市×エリアの俯瞰

In [ ]:
import matplotlib.colors as mcolors

# ヒートマップ用データの準備
df_heatmap = df_all.copy()
df_heatmap['ラベル'] = df_heatmap['都市'] + ' / ' + df_heatmap['エリア'] + ' (' + df_heatmap['タイプ'] + ')'
df_heatmap = df_heatmap.set_index('ラベル')[calc.FEATURE_COLUMNS]

# 正規化（列ごと）
df_heatmap_norm = (df_heatmap - df_heatmap.min()) / (df_heatmap.max() - df_heatmap.min() + 1e-10)

fig, ax = plt.subplots(figsize=(14, max(8, len(df_heatmap_norm) * 0.5)))
im = ax.imshow(df_heatmap_norm.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(calc.FEATURE_COLUMNS)))
ax.set_xticklabels(calc.FEATURE_COLUMNS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(df_heatmap_norm)))
ax.set_yticklabels(df_heatmap_norm.index, fontsize=8)

# セルに値を表示
for i in range(len(df_heatmap_norm)):
    for j in range(len(calc.FEATURE_COLUMNS)):
        val = df_heatmap.iloc[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=6)

plt.colorbar(im, ax=ax, label='正規化値')
ax.set_title('全分析結果ヒートマップ（正規化）', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 6.3 劉の類型化との対照まとめ

In [ ]:
# 市全域の指標と劉の4類型を対照
fig = plot_liu_typology_comparison(
    city_metrics,
    liu_type_profiles,
    title="市全域道路ネットワーク vs 劉（2016）4類型",
)
plt.show()

### 6.4 主要な知見のまとめ

In [ ]:
print("="*60)
print("主要な知見のまとめ")
print("="*60)

print("\n【仮説1: 沿岸部 vs 内陸部】")
print("沿岸部は内陸部と比べて回遊性が低く迂回性が高いか?")
for name in cities:
    if coastal_metrics[name] and inland_metrics[name]:
        alpha_diff = coastal_metrics[name]['alpha_index'] - inland_metrics[name]['alpha_index']
        circ_diff = coastal_metrics[name]['avg_circuity_A'] - inland_metrics[name]['avg_circuity_A']
        print(f"  {name}: alpha差={alpha_diff:+.4f}, circuity差={circ_diff:+.4f}")

print("\n【仮説2: 駅前地区の回遊性】")
print("都市規模が異なる3市の駅前地区の回遊性に差異があるか?")
for name in stations:
    alpha = station_metrics_walk[name]['alpha_index']
    deg = station_metrics_walk[name]['degree_centrality_mean']
    print(f"  {name}: alpha={alpha:.4f}, degree_centrality={deg:.4f}")

print("\n【仮説3: 細街路の回遊性寄与】")
print("細街路の寄与度は都市の性格により異なるか?")
for name in stations:
    alpha_effect = fine_street_effect[name]['alpha_index']
    print(f"  {name}: alpha_index への細街路効果={alpha_effect:+.4f}")

print("\n【仮説4: 都市固有のネットワーク特性】")
print("8指標による多面的評価で都市固有の特性を識別できるか?")
print("→ 上記のレーダーチャート・ヒートマップを参照")

## Section 7: 結果保存

### 7.1 指標データをCSV保存

In [ ]:
# 保存先ディレクトリの作成
data_dir = os.path.join(PROJECT_ROOT, 'data')
results_dir = os.path.join(PROJECT_ROOT, 'docs', 'results')
os.makedirs(data_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# 全結果CSVの保存
csv_path = os.path.join(data_dir, 'exp001_all_metrics.csv')
df_all.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"保存: {csv_path}")

# 市全域指標
csv_path = os.path.join(data_dir, 'exp001_city_metrics.csv')
df_city_metrics.to_csv(csv_path, encoding='utf-8-sig')
print(f"保存: {csv_path}")

# 細街路効果
csv_path = os.path.join(data_dir, 'exp001_fine_street_effect.csv')
df_fine_effect.to_csv(csv_path, encoding='utf-8-sig')
print(f"保存: {csv_path}")

### 7.2 地図をHTML保存

In [ ]:
# 駅周辺ネットワーク地図のHTML保存
for name in stations:
    m = plot_network_on_map(
        station_graphs_walk[name],
        title=f"{name}周辺{STATION_RADIUS}m 歩行者ネットワーク",
        metrics=station_metrics_walk[name],
        color="blue",
        weight=2,
    )
    html_path = os.path.join(
        results_dir,
        f'exp001_{name}_walk_network.html'
    )
    m.save(html_path)
    print(f"保存: {html_path}")

# 沿岸部ネットワーク地図のHTML保存
for name in cities:
    if coastal_graphs.get(name) is not None and coastal_graphs[name].number_of_nodes() > 0:
        G_orig = city_graphs[name]
        coastal_node_ids = list(coastal_graphs[name].nodes())
        valid_ids = [n for n in coastal_node_ids if n in G_orig.nodes()]
        if valid_ids:
            G_coastal_orig = G_orig.subgraph(valid_ids).copy()
            m = plot_network_on_map(
                G_coastal_orig,
                title=f"{name} 沿岸部",
                metrics=coastal_metrics[name],
                color="red",
            )
            html_path = os.path.join(
                results_dir,
                f'exp001_{name}_coastal_network.html'
            )
            m.save(html_path)
            print(f"保存: {html_path}")

### 7.3 グラフ画像をPNG保存

In [ ]:
# レーダーチャート
fig = plot_radar_chart(city_metrics, title="市全域: 3都市8指標レーダーチャート")
fig.savefig(
    os.path.join(results_dir, 'exp001_city_radar_chart.png'),
    dpi=150, bbox_inches='tight'
)
plt.close(fig)
print("保存: exp001_city_radar_chart.png")

# 3観点別棒グラフ
fig = plot_comparison_bars(city_metrics, title="市全域: 劉の3観点別指標比較")
fig.savefig(
    os.path.join(results_dir, 'exp001_city_comparison_bars.png'),
    dpi=150, bbox_inches='tight'
)
plt.close(fig)
print("保存: exp001_city_comparison_bars.png")

# 劉の類型対照レーダーチャート
fig = plot_liu_typology_comparison(
    city_metrics, liu_type_profiles,
    title="市全域 vs 劉（2016）4類型"
)
fig.savefig(
    os.path.join(results_dir, 'exp001_liu_typology_comparison.png'),
    dpi=150, bbox_inches='tight'
)
plt.close(fig)
print("保存: exp001_liu_typology_comparison.png")

# 駅周辺 vs 劉の類型
fig = plot_liu_typology_comparison(
    station_metrics_walk, liu_type_profiles,
    title="駅周辺歩行者ネットワーク vs 劉（2016）4類型"
)
fig.savefig(
    os.path.join(results_dir, 'exp001_station_liu_comparison.png'),
    dpi=150, bbox_inches='tight'
)
plt.close(fig)
print("保存: exp001_station_liu_comparison.png")

print("\n全ての結果を保存しました。")